# ProtSpace — Embedding Annotation Transfer (EAT)

This notebook demonstrates **Embedding Annotation Transfer (EAT)** with `protspace transfer`.
For each query protein that lacks an annotation value, the command finds the closest
annotated reference protein in pLM embedding space and transfers its label, together
with a reliability index in [0, 1]. The exact confidence formula depends on `--metric` and `--k`:

- Default (`--metric cosine`, `--k 1`): `confidence = clamp(1 - cosine_distance, 0, 1)` (cosine distance in [0, 2]). Cosine is the default because the confidence is naturally bounded and directly interpretable.
- `--metric euclidean` (`--k 1`): `confidence = 0.5 / (0.5 + distance)` — the published goPredSim transform, calibrated for ProtT5; on embedding spaces with larger raw distances treat it as a ranking rather than a calibrated probability.
- `--k > 1`: the goPredSim mean reliability — `(1/m) * sum` of the per-neighbour similarity over the `k` nearest neighbours carrying the chosen label, where `m = min(k, number of references)`. Because of this normalization, values are not comparable across different `--k` settings.

The method follows the goPredSim approach introduced in:

- Littmann et al., *Sci Rep* 2021 — [DOI 10.1038/s41598-020-80786-0](https://doi.org/10.1038/s41598-020-80786-0)
- Heinzinger et al., *NAR Genom Bioinform* 2022 — [DOI 10.1093/nargab/lqac043](https://doi.org/10.1093/nargab/lqac043)

Distances are computed in the original high-dimensional embedding space (HDF5),
not in any 2-D/3-D projection. The curated source column is left untouched;
results are written as `COL__pred_value`, `COL__pred_confidence`, and
`COL__pred_source` columns in the bundle's annotations table. `COL__pred_source`
is the reference protein the label was transferred from — provenance for a
connector line or "transferred from &lt;neighbour&gt;" tooltip in the web frontend.

📚 [GitHub](https://github.com/tsenoner/protspace) · [CLI Reference](https://github.com/tsenoner/protspace/blob/main/docs/cli.md#protspace-transfer) · [Annotation Reference](https://github.com/tsenoner/protspace/blob/main/docs/annotations.md#prediction-overlay-columns-eat-transfer)

## Install

In [ ]:
!pip install protspace

## Run transfer

Transfer the `protein_category` annotation from annotated reference proteins
(filtered to those with `protein_category` containing `neurotoxin`) to
unannotated query proteins whose IDs start with `TRINITY_`.

Adjust `-b`, `-e`, `-t`, `--query-id-prefix`, and `--reference-where` to match your data.

In [ ]:
!protspace transfer \
  -b results.parquetbundle \
  -e embeddings.h5:prot_t5 \
  -t protein_category \
  -o results.parquetbundle \
  --query-id-prefix TRINITY_ \
  --reference-where 'protein_category~neurotoxin'

## Read predictions back

Load the updated bundle and inspect the `*__pred_*` columns for the transferred annotations.

In [ ]:
import io
import pyarrow.parquet as pq
from protspace.data.io.bundle import read_bundle

parts, _ = read_bundle("results.parquetbundle")
df = pq.read_table(io.BytesIO(parts[0])).to_pandas()

# Show the curated column next to its predicted value / confidence / source
# overlay for the proteins that received a transferred label.
pred_value_cols = [c for c in df.columns if c.endswith("__pred_value")]
if pred_value_cols:
    col = pred_value_cols[0][: -len("__pred_value")]
    overlay = [
        c
        for c in (col, f"{col}__pred_value", f"{col}__pred_confidence", f"{col}__pred_source")
        if c in df.columns
    ]
    display(df.loc[df[f"{col}__pred_value"].notna(), overlay].head())
else:
    display(df.head())